# 04 — Detektor Drift Streaming: $W_1$ + CUSUM (Paper 3, Tahap T2)

**Dijalankan di SageMaker.** Memformalkan PoC Paper 1
(`../../unswnb-15/notebooks/30_drift_detector_poc.ipynb`): detektor *drift* berbasis
**jendela-geser Wasserstein $W_1$ + CUSUM** yang menyala saat distribusi trafik bergeser
jauh dari acuan latih. Di Paper 3, alarm drift = **sinyal kapan kelas serangan baru mungkin
muncul** → memicu jalur open-set (nb 02) + clustering (nb 03).

**Beda dgn PoC 30_:** (1) kalibrasi ambang lebih rapi (garis dasar dari segmen acuan,
bukan seluruh skor); (2) laporkan **delay deteksi** (jarak alarm dari titik transisi);
(3) uji sensitivitas $W$ (ukuran jendela) — trade-off deteksi cepat vs false alarm.

**Alur:** bentuk stream berurutan **CIC → UNSW** (domain shift terkontrol; opsional +AWS
bila hasil live tersedia) → z-score gabungan → sliding $W_1$ vs acuan CIC → CUSUM →
deteksi + delay + plot.

**Output** (→ S3 `evolusion/drift/`): `drift_results.json`, `drift_stream_<cfg>.png`,
`drift_sensitivity.csv`.

> Skor $S_t = \frac{1}{d}\sum_j W_1(\mathcal{W}_t^{(j)}, \mathcal{D}_{ref}^{(j)})$ (documentation.md §4).
> Semua angka dari eksekusi nyata.

In [ ]:
import importlib.util as u, sys, subprocess
need=[m for m in ('scipy','pandas','numpy','matplotlib','boto3','scikit-learn') if u.find_spec(m.replace('scikit-learn','sklearn')) is None]
if need: subprocess.run([sys.executable,'-m','pip','install','-q',*need],check=True)
print('setup ok' if not need else f'installed {need}')
print('=== SEL 0 (setup) SELESAI ===')

In [ ]:
import os, json, glob, datetime
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from scipy.stats import wasserstein_distance
plt.rcParams.update({'figure.dpi':120,'font.size':10,'axes.grid':True,'grid.alpha':0.3})
S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717')
S3_PREFIX='evolusion'; REGION=os.environ.get('AWS_REGION','ap-southeast-1')
OUTDIR='drift_out'; os.makedirs(OUTDIR,exist_ok=True)
CANON=['duration','fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','fwd_mean','bwd_mean','src_load','dst_load']
SEED=42; rng=np.random.default_rng(SEED)
NREF=5000; NSEG=6000     # ukuran acuan & tiap segmen stream
RESULTS={'generated':datetime.datetime.utcnow().isoformat()+'Z','features':CANON,'seed':SEED}
def savefig(n): p=os.path.join(OUTDIR,n); plt.savefig(p,bbox_inches='tight'); plt.close(); print(' saved',p); return p
def first(paths):
    for p in paths:
        h=sorted(glob.glob(p))
        if h: return h[0]
    return None
print('=== SEL 1 (config) SELESAI ===')

## 2. Loader 9-fitur CIC & UNSW (identik nb 01–03, tanpa label)

In [ ]:
def load_cic():
    p=first(['../../CICDDoS2018/data/file_100.csv','../../CICDDoS2018/data/file_*.csv'])
    if not p: print('CIC csv tak ada'); return None
    c=pd.read_csv(p, low_memory=False); c.columns=c.columns.str.strip()
    cm={'duration':'Flow Duration','fwd_pkts':'Tot Fwd Pkts','bwd_pkts':'Tot Bwd Pkts','fwd_bytes':'TotLen Fwd Pkts',
        'bwd_bytes':'TotLen Bwd Pkts','fwd_mean':'Fwd Pkt Len Mean','bwd_mean':'Bwd Pkt Len Mean','src_load':'Flow Byts/s','dst_load':'Bwd Pkts/s'}
    if not all(v in c.columns for v in cm.values()): print('CIC kolom kurang'); return None
    d=pd.DataFrame({k:pd.to_numeric(c[cm[k]],errors='coerce') for k in CANON})
    return d.replace([np.inf,-np.inf],np.nan).dropna()

def _pick_unsw():
    cands=[]
    for pat in ['../data/UNSW_NB15_*set.csv','../../unswnb-15/data/UNSW_NB15_*set.csv']:
        cands+=sorted(glob.glob(pat))
    cands=list(dict.fromkeys(cands))
    if not cands: return None
    best,best_n=None,-1
    for p in cands:
        try: n=sum(1 for _ in open(p,'r',errors='ignore'))-1
        except Exception: n=-1
        if n>best_n: best,best_n=p,n
    print(f'    UNSW dipakai: {os.path.basename(best)} (~{best_n} record)'); return best

def load_uns():
    p=_pick_unsw()
    if not p: print('UNSW csv tak ada'); return None
    u2=pd.read_csv(p); need=['dur','spkts','dpkts','sbytes','dbytes','smean','dmean','sload','dload']
    if not all(x in u2.columns for x in need): print('UNSW kolom kurang'); return None
    d=pd.DataFrame({'duration':pd.to_numeric(u2['dur'],errors='coerce')*1e6,'fwd_pkts':u2['spkts'],'bwd_pkts':u2['dpkts'],
                    'fwd_bytes':u2['sbytes'],'bwd_bytes':u2['dbytes'],'fwd_mean':u2['smean'],'bwd_mean':u2['dmean'],
                    'src_load':pd.to_numeric(u2['sload'],errors='coerce')/8.0,
                    'dst_load':pd.to_numeric(u2['dpkts'],errors='coerce')/pd.to_numeric(u2['dur'],errors='coerce').replace(0,np.nan)})
    return d.replace([np.inf,-np.inf],np.nan).dropna()

def try_load_aws():
    """Opsional: 9-fitur hasil AWS (nb/extractor) bila sudah ada lokal/S3. Tak wajib."""
    p=first(['aws_flows/*.csv','../aws/results/*_flows.csv'])
    if not p: return None
    try:
        a=pd.read_csv(p); a=a[[c for c in CANON if c in a.columns]]
        return a.replace([np.inf,-np.inf],np.nan).dropna() if set(CANON)<=set(a.columns) else None
    except Exception: return None

cic=load_cic(); uns=load_uns(); aws=try_load_aws()
print('CIC',None if cic is None else len(cic),'| UNSW',None if uns is None else len(uns),
      '| AWS',None if aws is None else len(aws))
print('=== SEL 2 (loader) SELESAI ===')

## 3. Bentuk stream berurutan + acuan latih + z-score gabungan

In [ ]:
def samp(df,n): return df.iloc[rng.permutation(len(df))[:min(n,len(df))]].reset_index(drop=True)

ref = samp(cic, NREF)                       # acuan = domain latih awal (CIC)
segs=[('CIC',samp(cic,NSEG)),('UNSW',samp(uns,NSEG))]
if aws is not None and len(aws)>=500: segs.append(('AWS',samp(aws,min(NSEG,len(aws)))))
stream=pd.concat([s for _,s in segs],ignore_index=True)
labels_seg=[]; bounds=[]; c=0
for nm,s in segs:
    labels_seg+= [nm]*len(s); c+=len(s); bounds.append(c)
bounds=bounds[:-1]  # titik transisi (akhir segmen terakhir bukan transisi)
# z-score gabungan (fit pd ref+stream) agar W1 setara antar-fitur
allX=pd.concat([ref,stream],ignore_index=True); mu=allX.mean(); sd=allX.std().replace(0,1)
refz=((ref-mu)/sd)[CANON]; strz=((stream-mu)/sd)[CANON]
RESULTS['stream']={'segments':[nm for nm,_ in segs],'seg_sizes':[len(s) for _,s in segs],
                   'transitions':bounds,'n_ref':len(ref),'n_stream':len(strz)}
print('segmen',RESULTS['stream']['segments'],'ukuran',RESULTS['stream']['seg_sizes'],'transisi@',bounds)
print('=== SEL 3 (stream) SELESAI ===')

## 4. Sliding-window $W_1$ + CUSUM + delay deteksi

In [ ]:
def drift_score(win_z):
    return float(np.mean([wasserstein_distance(win_z[c].values, refz[c].values) for c in CANON]))

def run_detector(W, STEP=250, k_sigma=0.5, h_sigma=5.0, thr_sigma=3.0):
    centers=[]; scores=[]
    for start in range(0, len(strz)-W+1, STEP):
        centers.append(start+W//2); scores.append(drift_score(strz.iloc[start:start+W]))
    scores=np.array(scores); centers=np.array(centers)
    # garis dasar dari jendela yg SELURUHNYA di segmen acuan pertama (< transisi pertama - W)
    base_end = (bounds[0]-W) if bounds else len(strz)
    base_mask = centers < max(base_end, W)
    if base_mask.sum()<3: base_mask=centers < (bounds[0] if bounds else len(strz)//2)
    mu0=float(scores[base_mask].mean()); sd0=float(scores[base_mask].std() or 1e-6)
    thr=mu0+thr_sigma*sd0
    # CUSUM
    k=k_sigma*sd0; cusum=np.zeros(len(scores)); cc=0.0
    for i,s in enumerate(scores):
        cc=max(0.0, cc+(s-mu0)-k); cusum[i]=cc
    h=h_sigma*sd0
    a_w1=centers[scores>thr]; a_cs=centers[cusum>h]
    first_w1=int(a_w1[0]) if len(a_w1) else None
    first_cs=int(a_cs[0]) if len(a_cs) else None
    # delay deteksi = alarm pertama SETELAH transisi pertama - posisi transisi (flow)
    delay=None
    if bounds and first_cs is not None:
        after=[x for x in a_cs if x>=bounds[0]]
        if after: delay=int(after[0]-bounds[0])
    return dict(W=W,STEP=STEP,mu0=round(mu0,4),sd0=round(sd0,4),thr=round(thr,4),cusum_h=round(h,4),
                first_alarm_w1=first_w1,first_alarm_cusum=first_cs,detect_delay_flows=delay), centers, scores, cusum, thr, h

# konfigurasi utama
det, centers, scores, cusum, thr, h = run_detector(W=1000, STEP=250)
RESULTS['detector']=det
print(f"[detector W=1000] mu0={det['mu0']} thr={det['thr']} | alarm W1@{det['first_alarm_w1']} "
      f"CUSUM@{det['first_alarm_cusum']} | transisi@{bounds} delay={det['detect_delay_flows']} flow")
print('=== SEL 4 (detektor) SELESAI ===')

## 5. Plot + uji sensitivitas ukuran jendela W

In [ ]:
# plot skor drift + CUSUM
fig,ax=plt.subplots(2,1,figsize=(9,6),sharex=True)
ax[0].plot(centers,scores,'-o',ms=3,color='#4C72B0',label='skor drift $S_t$')
ax[0].axhline(thr,ls='--',color='#C44E52',label='ambang 3$\\sigma$')
for b in bounds: ax[0].axvline(b,ls=':',color='gray')
ax[0].set_ylabel('$S_t$'); ax[0].set_title('Streaming drift: $W_1$ jendela-geser vs acuan CIC'); ax[0].legend(fontsize=8)
ax[1].plot(centers,cusum,'-',color='#DD8452',label='CUSUM'); ax[1].axhline(h,ls='--',color='#C44E52',label='ambang $h$')
for b in bounds: ax[1].axvline(b,ls=':',color='gray')
ax[1].set_xlabel('indeks flow (waktu stream)'); ax[1].set_ylabel('CUSUM'); ax[1].legend(fontsize=8)
plt.tight_layout(); savefig('drift_stream_W1000.png')

# sensitivitas W: deteksi cepat vs false alarm (alarm sebelum transisi pertama = false)
sens=[]
for W in [500,1000,2000]:
    d2,cen2,sc2,cu2,th2,h2=run_detector(W=W,STEP=250)
    n_fa=int(((cen2<(bounds[0] if bounds else 0)) & (sc2>th2)).sum()) if bounds else None  # false alarm W1 di segmen acuan
    sens.append({'W':W,'thr':d2['thr'],'first_alarm_cusum':d2['first_alarm_cusum'],
                 'detect_delay_flows':d2['detect_delay_flows'],'false_alarm_w1_pre':n_fa})
sens_df=pd.DataFrame(sens); import IPython.display as ipd
print('Sensitivitas W (delay vs false alarm):'); ipd.display(sens_df)
sens_df.to_csv(os.path.join(OUTDIR,'drift_sensitivity.csv'),index=False)
RESULTS['sensitivity']=sens
print('=== SEL 5 (plot + sensitivitas) SELESAI ===')

## 6. Simpan + UPLOAD S3

In [ ]:
jp=os.path.join(OUTDIR,'drift_results.json'); json.dump(RESULTS,open(jp,'w'),indent=2); print('tersimpan',jp)
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for fn in sorted(os.listdir(OUTDIR)):
        if fn.endswith(('.json','.png','.csv')): s3.upload_file(os.path.join(OUTDIR,fn),S3_BUCKET,f'{S3_PREFIX}/drift/{fn}'); up+=1
    print(f'upload {up} artefak -> s3://{S3_BUCKET}/{S3_PREFIX}/drift/')
except Exception as e: print('upload gagal:',e)
print('=== SEL 6 (simpan + upload) SELESAI ===')
print('SELESAI T2. CUSUM menyala dekat titik transisi (delay kecil) = detektor drift valid. '
      'Alarm drift -> memicu jalur open-set (nb 02) + clustering (nb 03) di loop closed-loop.')